# 10주차 랩 · 전력 고객의 사용 패턴 묶기

**⏱ 60분** · 포르투갈 전력 고객 370곳의 2014년 시간별 사용량 · 개인 제출

## 🎯 이번 랩의 질문

교재에서는 RFM으로 고객을 묶어 재구매 캠페인 대상을 정했습니다. 이번에는 포르투갈의 한 전력 회사 고객이 하루 중 언제 전기를 쓰는지로 고객을 묶습니다. 전력 회사는 저녁(18~21시)에 몰리는 사용량을 줄이려 합니다. 그 시간대 사용을 줄이면 요금을 깎아 주는 **수요반응 프로그램**을 일부 고객에게 먼저 제안할 계획입니다.

> **고객을 사용 패턴에 따라 묶고, 저녁 수요반응 프로그램은 어느 무리에 먼저 제안할까요?**

원자료에는 기록이 비어 있는 고객과 시계가 바뀐 시각이 섞여 있습니다. 먼저 자료를 정리한 뒤, 교재의 패턴을 다시 씁니다.

| 패턴 | 교재 | 이번 랩 |
|---|---|---|
| 분석에 쓸 수 없는 행·열 빼기 | 준비 셀(취소 주문·0 이하 줄 빼기) | 과제 1 |
| 대상별 요약표 만들고 척도 맞추기 | 1·2절 | 과제 2 |
| `KMeans` 학습과 군집 크기 | 3절 | 과제 3 |
| k에 따른 WSS | 4절 | 과제 4 |
| Ward 계층 군집 | 5절 | 과제 5 |
| 군집별 요약으로 특징 읽기 | 6절 | 과제 6 |
| 군집 하나 골라 결정 | 7절 | 과제 7 |

### 진행 방법

| 표시 | 뜻 | 할 일 |
|---|---|---|
| ▶ | 제공 코드 | 그대로 실행합니다 |
| 📝 | 과제 | `None` 자리를 코드로 바꿔 채웁니다. 변수 이름은 바꾸지 않습니다 |
| ✅ | 확인 셀 | 고치지 말고 실행만 합니다 |
| 💬 | 결과 읽기 | 결과에서 확인할 숫자를 짚습니다 |

- 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다.
- 확인 셀은 변수의 형식만 봅니다. "형식 확인 완료"는 값이 맞다는 뜻이 아닙니다. 값은 제출 후 채점합니다.

## ▶ 준비

UCI에 공개된 포르투갈 전력 고객 370곳의 사용 기록(ElectricityLoadDiagrams20112014)에서 2014년 한 해를 씁니다. 원자료는 15분 간격 기록이고, 여기서는 시간 단위 사용량(kWh)으로 합쳐 두었습니다. `df`는 **한 행이 한 시간, 한 열이 고객 한 곳**인 표입니다(8,760시간 × 370곳). 열 이름 `MT_001` ~ `MT_370`이 고객 번호입니다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. 전력 사용 기록을 읽습니다. (20초 안팎)
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", 500)       # 표를 줄임표(...) 없이 끝까지 보여 줌
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", None)
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, fcluster
import os
import sys

for folder in (".", "..", "../..", "../../.."):
    if os.path.exists(os.path.join(folder, "balab.py")):
        sys.path.insert(0, folder)
        break
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py", "balab.py")

from balab import load, checker, summary, check_format
check = checker("week10")

df = load("electricity")
print("준비 완료!", df.shape)
df.iloc[:5, :5]

✅ `준비 완료! (8760, 370)`이 나오면 됩니다. 표에는 사용량이 0인 칸이 있습니다. 1년 중 많은 시간이 0인 고객은 계약 전이거나 계량 기록이 없는 고객입니다.

---
## 📝 과제 1 · 기록 정리

분석 전에 쓸 수 없는 기록을 뺍니다.

- **기록이 부족한 고객:** 고객마다 사용량이 0인 시간의 비율을 구해 `zero_share`에 담습니다. 참·거짓 열의 평균은 참의 비율입니다. 그 비율이 0.1 이상인 고객의 열을 빼고 다시 `df`에 담습니다.
- **서머타임이 바뀐 두 시각:** 행 두 개를 `df`에서 뺍니다.
  - 2014-03-30 01:00: 시계가 1시에서 2시로 건너뛰어 기록이 0입니다.
  - 2014-10-26 01:00: 1시가 두 번 있어 두 시간 치가 합쳐져 있습니다.
- 남은 시간 수를 `n_hours`, 고객 수를 `n_customers`에 담습니다.

**결과 형식:** `n_hours`, `n_customers`는 정수

In [ ]:
# 📝 과제 1 — None 자리를 채우세요
# 힌트: (df == 0)의 열 평균이 0의 비율입니다. 조건에 맞는 열만 고를 때는 df.loc[:, 조건]을 씁니다. 행은 drop(index=[...])로 빼고, 시각은 pd.Timestamp("2014-03-30 01:00")처럼 적습니다. 행·열 수는 shape에 있습니다.
zero_share = None   # 고객별 0인 시간의 비율
df = None           # 정리한 사용량 표
n_hours = None      # 남은 시간 수
n_customers = None  # 남은 고객 수

In [ ]:
# ✅ 확인 — 과제 1 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(1, n_hours=n_hours, n_customers=n_customers)
else:
    check_format(1, n_hours=n_hours, n_customers=n_customers)

💬 **결과 읽기** — 빠진 고객은 몇 곳이고, 그 가운데 1년 내내 기록이 없는 고객은 몇 곳인가요?

---
## 📝 과제 2 · 고객별 하루 사용 모양

교재 1절에서 고객마다 한 행인 RFM 표를 만들고 2절에서 척도를 맞췄듯이, 고객마다 한 행인 요약표를 만들고 척도를 맞춥니다.

- 시각(0~23시)별로 묶어 평균을 낸 뒤 행과 열을 바꿔(`.T`), 고객 × 시각 평균 사용량 표 `profile`을 만듭니다. `df.index.hour`가 각 행의 시각입니다.
- 사용 규모가 큰 고객과 작은 고객을 같은 기준으로 비교하려고, `profile`을 고객마다 자기 하루 평균으로 나눈 표를 `shape`에 담습니다. 1이면 그 고객의 평균만큼, 2이면 평균의 두 배를 그 시각에 씁니다.
- `shape`의 행 수를 `n_rows`, 열 수를 `n_cols`에 담습니다.

**결과 형식:** `n_rows`, `n_cols`는 정수

In [ ]:
# 📝 과제 2 — None 자리를 채우세요
# 힌트: df.groupby(df.index.hour).mean()에 .T를 붙입니다. 고객마다 하루 평균은 profile.mean(axis=1)이고, 행마다 나눌 때는 div(나눌 값, axis=0)를 씁니다.
profile = None  # 고객 × 시각 평균 사용량
shape = None    # 고객 × 시각 사용 모양 (하루 평균 = 1)
n_rows = None   # shape의 행 수
n_cols = None   # shape의 열 수

In [ ]:
# ✅ 확인 — 과제 2 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(2, n_rows=n_rows, n_cols=n_cols)
else:
    check_format(2, n_rows=n_rows, n_cols=n_cols)

💬 **결과 읽기** — `MT_002`는 몇 시에 자기 하루 평균의 몇 배를 쓰나요? `profile.mean(axis=1)`로 고객마다 하루 평균 사용량도 보세요. 규모가 수십 배 차이 나는 고객도 `shape`에서는 같은 기준으로 비교됩니다.

---
## 📝 과제 3 · k-means로 세 무리

교재 3절처럼 `shape`를 k-means로 묶습니다. 군집 수는 3, `n_init=10`, `random_state=0`입니다.

- 학습한 모형을 `model`에 담습니다.
- 군집 크기를 큰 순서로 나열한 리스트를 `sizes`에 담습니다.

**결과 형식:** `sizes`는 정수 3개의 리스트(큰 순서)

In [ ]:
# 📝 과제 3 — None 자리를 채우세요
# 힌트: 교재 3절의 두 줄에서 X를 shape로, n_clusters를 3으로 바꿉니다. 크기는 value_counts()에 tolist()를 붙이면 큰 순서의 리스트가 됩니다.
model = None  # k = 3 k-means
sizes = None  # 군집 크기, 큰 순서

In [ ]:
# ✅ 확인 — 과제 3 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(3, sizes=sizes)
else:
    check_format(3, sizes=sizes)

💬 **결과 읽기** — 가장 큰 군집과 가장 작은 군집에는 고객이 몇 곳인가요?

---
## 📝 과제 4 · 군집 수에 따른 WSS

교재 4절처럼 k를 1부터 7까지 바꿔 WSS를 구해 `wss` 리스트에 담습니다(`n_init=10`, `random_state=0`).

**결과 형식:** `wss`는 숫자 7개의 리스트(k = 1~7 순서)

In [ ]:
# 📝 과제 4 — None 자리를 채우세요
# 힌트: 교재 4절의 for 반복에서 X를 shape로, range를 1~7로 바꿉니다.
wss = None  # k = 1~7의 WSS

In [ ]:
# ✅ 확인 — 과제 4 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(4, wss=(wss, 0.01))
else:
    check_format(4, wss=wss)

💬 **결과 읽기** — k = 2, 3, 4에서 WSS는 각각 몇 % 줄어드나요? 과제 3의 k = 3은 교재의 기준(20% 이상 줄어드는 마지막 k)과 맞나요?

---
## 📝 과제 5 · Ward 계층 군집

교재 5절처럼 `shape`로 Ward 계층 군집을 만들고 군집 3개로 자릅니다.

- `linkage` 결과를 `Z`, `fcluster`로 자른 군집 번호를 `labels_ward`에 담습니다.
- 군집 크기를 큰 순서로 나열한 리스트를 `sizes_ward`에 담습니다.

**결과 형식:** `sizes_ward`는 정수 3개의 리스트(큰 순서)

In [ ]:
# 📝 과제 5 — None 자리를 채우세요
# 힌트: 교재 5절의 linkage·fcluster 두 줄에서 X를 shape로, t를 3으로 바꿉니다. 크기는 과제 3과 같은 방법으로 셉니다.
Z = None           # Ward 합치기 기록
labels_ward = None # 3개로 자른 군집 번호
sizes_ward = None  # 군집 크기, 큰 순서

In [ ]:
# ✅ 확인 — 과제 5 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(5, sizes_ward=sizes_ward)
else:
    check_format(5, sizes_ward=sizes_ward)

💬 **결과 읽기** — 교차표에서 두 방법이 같은 무리로 묶은 고객은 몇 곳인가요?

---
## 📝 과제 6 · 군집별 저녁 사용 모양과 사용량 비중

교재 6절처럼 군집별로 요약합니다. 저녁 수요반응이 관심이므로 18~21시를 봅니다.

- `shape`를 k-means 군집(`model.labels_`)별로 평균 낸 표를 `centers`에 담습니다(군집 × 24시간).
- `centers`의 18·19·20·21시 열을 평균 낸 값 가운데 가장 큰 값을 `best_evening`에 담습니다.
- 고객별 1년 사용량 합계(`df.sum()`)를 군집별로 더해 전체 합계로 나눈 비중을 `share`, 그 가운데 가장 큰 값을 `top_share`에 담습니다.
- 아래 정답 셀의 그림으로 군집별 하루 모양을 봅니다.

**결과 형식:** `best_evening`, `top_share`는 숫자

In [ ]:
# 📝 과제 6 — None 자리를 채우세요
# 힌트: 교재 6절의 groupby에 model.labels_를 넣고 mean()을 붙입니다. 열은 [18, 19, 20, 21]처럼 고르고, 행마다 평균은 mean(axis=1)입니다. 사용량 비중은 groupby(model.labels_).sum()을 전체 합계로 나눕니다.
centers = None       # 군집 × 24시간 평균 모양
best_evening = None  # 18~21시 평균 모양값의 최댓값
share = None         # 군집별 연간 사용량 비중
top_share = None     # 가장 큰 사용량 비중

In [ ]:
# ✅ 확인 — 과제 6 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(6, best_evening=(best_evening, 0.01), top_share=(top_share, 0.001))
else:
    check_format(6, best_evening=best_evening, top_share=top_share)

💬 **결과 읽기** — 저녁 18~21시에 하루 평균보다 가장 많이 쓰는 군집은 평균의 몇 배를 쓰고, 연간 사용량의 몇 %를 차지하나요?

---
## 📝 과제 7 · 본인의 판단

1. 저녁 수요반응 프로그램을 먼저 제안할 군집 하나를 고르고, 과제 6의 숫자 두 개로 근거를 적으세요.
2. 고르지 않은 군집 하나는 왜 뒤로 미루는지 숫자 하나로 적으세요.

자동 채점은 하지 않습니다.

✏️ **나의 답:** (여기를 두 번 눌러 적습니다)

---
## ✅ 제출 전 확인

아래 셀을 실행해 과제 1~6의 작성 여부와 형식을 확인합니다. "형식 확인 완료"는 계산값이 맞다는 뜻이 아닙니다. 결과를 직접 읽고 과제 7을 작성해 제출합니다. 실제 값 채점은 제출 후 진행합니다.

In [ ]:
# ✅ 제출 전 확인 — 과제 1~6의 작성 여부와 형식
summary()